# Advanced: Multi-Strategy K-Anonymity Profiling with PAMOLA.CORE

**Goal**: Master all k-anonymity profiling strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Single QI combination analysis (detailed metrics)
- **Strategy 2**: Multiple QI combinations (comparative analysis)
- **Strategy 3**: DataFrame enrichment (add k-values as column)
- Calculate advanced privacy metrics (entropy, vulnerability)
- Analyze re-identification risks
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of k-anonymity concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 02_k_anonymity_profiler_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.anonymity import KAnonymityProfilerOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record patient dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 7 columns)
- Sample records (first 5 rows)
- Column statistics (types, ranges, unique counts)

**Note:** Generated data includes realistic quasi-identifiers and privacy vulnerabilities

In [ ]:
# Try to load from file first
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'k_anonymity_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    age_groups = ['18-25', '26-35', '36-45', '46-55', '56-65', '66-75', '76+']
    genders = ['M', 'F', 'Other']
    zip_codes = [f'100{i:02d}' for i in range(1, 21)]
    diagnoses = ['Type A', 'Type B', 'Type C', 'Type D', 'Type E', 'Type F']
    education = ['High School', 'Bachelor', 'Master', 'PhD', 'Other']
    
    df = pd.DataFrame({
        'patient_id': range(1, n_records + 1),
        'age_group': np.random.choice(age_groups, n_records, p=[0.15, 0.20, 0.25, 0.20, 0.12, 0.06, 0.02]),
        'gender': np.random.choice(genders, n_records, p=[0.48, 0.48, 0.04]),
        'zip_code': np.random.choice(zip_codes, n_records),
        'diagnosis': np.random.choice(diagnoses, n_records),
        'education': np.random.choice(education, n_records, p=[0.25, 0.35, 0.25, 0.10, 0.05]),
        'treatment_cost': np.random.randint(1000, 50000, n_records)
    })
    
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE QI_CONFIG**: Edit quasi-identifier sets for each strategy
   - `strategy1_qi`: Single combination (e.g., ['age_group', 'gender'])
   - `strategy2_qi`: Multiple combinations for comparison
   - `strategy3_qi`: Fields for enrichment mode
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each QI field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All quasi-identifiers must exist in dataset before executing strategies

In [ ]:
# QI configuration - CUSTOMIZE THESE!
QI_CONFIG = {
    "strategy1_qi": ["age_group", "gender"],
    "strategy2_qi": [
        ["age_group", "gender"],
        ["age_group", "zip_code"],
        ["gender", "education"]
    ],
    "strategy3_qi": ["age_group", "gender", "zip_code"]
}

# Validate all QI fields exist
print("📋 Validating QI configuration...\n")

all_qi_fields = set()
all_qi_fields.update(QI_CONFIG["strategy1_qi"])
for qi_set in QI_CONFIG["strategy2_qi"]:
    all_qi_fields.update(qi_set)
all_qi_fields.update(QI_CONFIG["strategy3_qi"])

for field in sorted(all_qi_fields):
    if field not in df.columns:
        raise ValueError(
            f"❌ Field '{field}' not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update QI_CONFIG"
        )
    print(f"  ✓ {field:15s}: {df[field].nunique()} unique values")

def create_config_kwargs():
    return {"dataset_name": "main_dataset"}

examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy k-anonymity profiling",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Single QI Combination Analysis

**How to use:**
- Analyzes one quasi-identifier combination in detail
- Run to execute comprehensive privacy profiling
- Monitor progress bar (6 steps)

**Key parameters:**
- `analysis_mode='ANALYZE'`: Generate metrics without modifying data
- `threshold_k=5`: Records with k<5 are vulnerable
- `export_metrics=True`: Save detailed JSON/CSV files

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- K-anonymity metrics (min k, mean k, vulnerable %)

**Note:** Creates detailed metrics and visualizations for privacy assessment

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: SINGLE QI COMBINATION ANALYSIS")
print("=" * 80 + "\n")

tracker_s1 = HierarchicalProgressTracker(
    total=6, description="Strategy 1: Single QI", unit="steps",
    track_memory=True, level=0
)

operation_s1 = KAnonymityProfilerOperation(
    quasi_identifiers=QI_CONFIG["strategy1_qi"],
    analysis_mode='ANALYZE',
    threshold_k=5,
    max_combinations=1,
    id_fields=['patient_id'],
    export_metrics=True,
    output_field_suffix='k_anon',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  QI: {', '.join(operation_s1.quasi_identifiers)}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_single_qi',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results
output_dir_s1 = task_dir / 'strategy1_single_qi' / 'output'
summary_files_s1 = sorted(
    list(output_dir_s1.glob('*summary*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)

if summary_files_s1:
    with open(summary_files_s1[0], 'r') as f:
        summary_s1 = json.load(f)
    
    first_qi = list(summary_s1.keys())[0]
    metrics_s1 = summary_s1[first_qi]
    
    print(f"\n📊 K-Anonymity Metrics:")
    print(f"  Min k: {metrics_s1['min_k']}")
    print(f"  Mean k: {metrics_s1['mean_k']:.2f}")
    print(f"  Vulnerable %: {metrics_s1['vulnerable_percent']:.2f}%")
    print(f"  Entropy: {metrics_s1['entropy']:.4f}")

## STRATEGY 2: Multiple QI Combinations

**How to use:**
- Analyzes multiple quasi-identifier combinations
- Compares privacy risks across different field sets
- Run to execute comparative privacy analysis

**Key parameters:**
- `quasi_identifier_sets`: List of QI combinations to analyze
- `max_combinations=10`: Limit number of combinations
- `analysis_mode='ANALYZE'`: Metrics without data modification

**What you'll see:**
- Configuration summary
- Progress: validation → analysis (per combination) → aggregate → save
- Completion time and status
- Comparative metrics across all combinations

**Note:** Best for understanding which field combinations pose highest privacy risks

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: MULTIPLE QI COMBINATIONS")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6, description="Strategy 2: Multiple QI", unit="steps",
    track_memory=True, level=0
)

operation_s2 = KAnonymityProfilerOperation(
    quasi_identifiers=[],
    quasi_identifier_sets=QI_CONFIG["strategy2_qi"],
    analysis_mode='ANALYZE',
    threshold_k=5,
    max_combinations=10,
    id_fields=['patient_id'],
    export_metrics=True,
    output_field_suffix='k_anon',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  QI combinations: {len(QI_CONFIG['strategy2_qi'])}")
for i, qi_set in enumerate(QI_CONFIG["strategy2_qi"], 1):
    print(f"    {i}. {', '.join(qi_set)}")

print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_multiple_qi',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_dir_s2 = task_dir / 'strategy2_multiple_qi' / 'output'
summary_files_s2 = sorted(
    list(output_dir_s2.glob('*summary*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)

if summary_files_s2:
    with open(summary_files_s2[0], 'r') as f:
        summary_s2 = json.load(f)
    
    print(f"\n📊 Comparative K-Anonymity:")
    for qi_name, metrics in summary_s2.items():
        print(f"\n  {qi_name}:")
        print(f"    Min k: {metrics['min_k']}")
        print(f"    Mean k: {metrics['mean_k']:.2f}")
        print(f"    Vulnerable %: {metrics['vulnerable_percent']:.2f}%")

## STRATEGY 3: DataFrame Enrichment

**How to use:**
- Adds k-anonymity values as new column to DataFrame
- Useful for downstream filtering or analysis
- Run to enrich data with privacy metrics

**Key parameters:**
- `analysis_mode='ENRICH'`: Add k-values to DataFrame
- `output_field_suffix='k_anon'`: Suffix for new column
- Creates column like: `KA_ag_ge_zi_k_anon`

**What you'll see:**
- Configuration summary
- Progress: validation → k-value calculation → enrichment → save
- Completion time and status
- Preview of enriched DataFrame

**Note:** Original data preserved, new k-anonymity column added

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: DATAFRAME ENRICHMENT")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6, description="Strategy 3: Enrichment", unit="steps",
    track_memory=True, level=0
)

operation_s3 = KAnonymityProfilerOperation(
    quasi_identifiers=QI_CONFIG["strategy3_qi"],
    analysis_mode='ENRICH',
    threshold_k=5,
    max_combinations=1,
    id_fields=['patient_id'],
    export_metrics=False,
    output_field_suffix='k_anon',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  QI: {', '.join(operation_s3.quasi_identifiers)}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_enrichment',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load enriched data
output_dir_s3 = task_dir / 'strategy3_enrichment' / 'output'
enriched_files_s3 = sorted(
    list(output_dir_s3.glob('enriched_data*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)

if enriched_files_s3:
    enriched_df_s3 = pd.read_csv(enriched_files_s3[0])
    k_col = [col for col in enriched_df_s3.columns if 'k_anon' in col]
    
    if k_col:
        k_col_name = k_col[0]
        print(f"\n📊 Enriched DataFrame:")
        print(f"  New column: {k_col_name}")
        print(f"  Min k: {enriched_df_s3[k_col_name].min()}")
        print(f"  Max k: {enriched_df_s3[k_col_name].max()}")
        print(f"  Mean k: {enriched_df_s3[k_col_name].mean():.2f}")
        
        print(f"\n🔍 Sample (first 10 rows):")
        display_cols = ['patient_id'] + QI_CONFIG["strategy3_qi"] + [k_col_name]
        existing_cols = [col for col in display_cols if col in enriched_df_s3.columns]
        print(enriched_df_s3[existing_cols].head(10))

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Privacy risk comparison
- Best use case recommendations

**Note:** Fastest strategy isn't always best - consider analysis depth needed

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Single QI):    {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Multiple QI):  {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Enrichment):   {elapsed_s3:6.2f}s")
print(f"  Total:                     {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n💡 Recommendations:")
print(f"  - Use Strategy 1 for focused privacy assessment")
print(f"  - Use Strategy 2 to identify riskiest field combinations")
print(f"  - Use Strategy 3 when k-values needed for filtering")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Reviews all strategies automatically
- Displays newest files only

**What you'll see (per strategy):**
1. **Metrics JSON**: K-anonymity metrics, performance
2. **Summary Data**: Privacy assessment, vulnerabilities
3. **Visualizations**: Charts displayed inline (latest batch)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
strategy_dirs = [
    ('strategy1_single_qi', 'Strategy 1: Single QI Analysis'),
    ('strategy2_multiple_qi', 'Strategy 2: Multiple QI Combinations'),
    ('strategy3_enrichment', 'Strategy 3: DataFrame Enrichment')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics Summary (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        if metrics_files:
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]
            target_files = filtered if filtered else metrics_files
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            
            print(f"\n📄 METRICS: {target_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(target_files[0], 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                if 'k_anonymity' in metrics:
                    k_anon_metrics = metrics['k_anonymity']
                    print(f"   K-Anonymity Combinations: {len(k_anon_metrics)}")
                    print(f"\n   📊 Summary by QI Combination:")
                    
                    for qi_name, qi_metrics in list(k_anon_metrics.items())[:5]:  # Show first 5
                        print(f"      • {qi_name}:")
                        print(f"        - Min k              : {qi_metrics.get('min_k', 0)}")
                        print(f"        - Mean k             : {qi_metrics.get('mean_k', 0):.2f}")
                        print(f"        - Vulnerable records : {qi_metrics.get('vulnerable_percent', 0):.2f}%")
                        print(f"        - Entropy            : {qi_metrics.get('entropy', 0):.2f}")
                    
                    if len(k_anon_metrics) > 5:
                        print(f"      ... and {len(k_anon_metrics) - 5} more combinations")
                        
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. K-Anonymity Summary (from output/)
    output_dir = strategy_dir / 'output'
    if output_dir and output_dir.exists():
        summary_files = sorted(
            list(output_dir.glob('*_summary_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if summary_files:
            print(f"\n📊 K-ANONYMITY SUMMARY: {summary_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(summary_files[0], 'r') as f:
                    summary_data = json.load(f)
                
                print(f"   Total QI combinations: {len(summary_data)}")
                print(f"\n   Detailed Breakdown:")
                
                # Sort by vulnerable_percent descending
                sorted_items = sorted(
                    summary_data.items(), 
                    key=lambda x: x[1].get('vulnerable_percent', 0), 
                    reverse=True
                )
                
                for qi_name, qi_data in sorted_items[:5]:  # Show top 5 by vulnerability
                    print(f"\n      🔍 {qi_name}:")
                    print(f"         Fields               : {qi_data.get('fields', 'N/A')}")
                    print(f"         Min k                : {qi_data.get('min_k', 0)}")
                    print(f"         Mean k               : {qi_data.get('mean_k', 0):.2f}")
                    print(f"         Vulnerable %         : {qi_data.get('vulnerable_percent', 0):.2f}%")
                    print(f"         Entropy              : {qi_data.get('entropy', 0):.4f}")
                
                if len(summary_data) > 5:
                    print(f"\n      ... and {len(summary_data) - 5} more combinations")
                    
            except Exception as e:
                print(f"   ⚠️  Error reading summary: {e}")
    
    # 3. Vulnerable Records Analysis (from output/)
    if output_dir and output_dir.exists():
        vuln_files = sorted(
            list(output_dir.glob('*vulnerable_records_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if vuln_files:
            print(f"\n🔒 VULNERABLE RECORDS: {vuln_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(vuln_files[0], 'r') as f:
                    vuln_data = json.load(f)
                
                print(f"   QI combinations analyzed: {len(vuln_data)}")
                
                # Calculate total statistics
                total_vulnerable = sum(
                    qi_data.get('total_vulnerable', 0) 
                    for qi_data in vuln_data.values()
                )
                total_groups = sum(
                    qi_data.get('vulnerable_groups', 0) 
                    for qi_data in vuln_data.values()
                )
                
                print(f"   Total vulnerable records: {total_vulnerable:,}")
                print(f"   Total vulnerable groups : {total_groups:,}")
                
                # Show details for each QI combination
                print(f"\n   Details by QI Combination:")
                
                # Sort by total_vulnerable descending
                sorted_vuln = sorted(
                    vuln_data.items(), 
                    key=lambda x: x[1].get('total_vulnerable', 0), 
                    reverse=True
                )
                
                for qi_name, qi_vuln in sorted_vuln[:5]:  # Show top 5 by vulnerability
                    print(f"\n      🎯 {qi_name}:")
                    print(f"         Vulnerable records   : {qi_vuln.get('total_vulnerable', 0):,}")
                    print(f"         Vulnerable %         : {qi_vuln.get('vulnerable_percent', 0):.2f}%")
                    print(f"         Vulnerable groups    : {qi_vuln.get('vulnerable_groups', 0):,}")
                    
                    # Show sample IDs
                    sample_ids = qi_vuln.get('sample_ids', [])
                    if sample_ids:
                        ids_str = ', '.join(map(str, sample_ids[:10]))
                        if len(sample_ids) > 10:
                            ids_str += f" ... (+{len(sample_ids) - 10} more)"
                        print(f"         Sample IDs           : {ids_str}")
                
                if len(vuln_data) > 5:
                    print(f"\n      ... and {len(vuln_data) - 5} more combinations")
                    
            except Exception as e:
                print(f"   ⚠️  Error reading vulnerable records: {e}")
    
    # 4. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [f for f in viz_files if abs(f.stat().st_mtime - latest_time) < 10]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:
                print(f"\n   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Privacy Metrics

**How to use:**
- Run AFTER all strategies complete
- Calculates comprehensive privacy metrics

**What you'll see:**
- Original data: min k, avg k, vulnerable groups
- Comparison across QI combinations
- Risk level assessment

**Privacy targets:**
- Min k ≥ 5: Basic protection
- Min k ≥ 10: Strong protection
- Zero vulnerable groups: Ideal

**Note:** Requires result data from strategy execution cells

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS ANALYSIS")
print("=" * 80 + "\n")

def calculate_k_anonymity(df, quasi_identifiers):
    group_sizes = df.groupby(quasi_identifiers).size()
    return {
        'min_k': int(group_sizes.min()),
        'avg_k': float(group_sizes.mean()),
        'vulnerable_groups': int((group_sizes < 5).sum())
    }

try:
    print(f"📊 ORIGINAL DATA PRIVACY:")
    print("-" * 80)
    
    for i, qi_set in enumerate(QI_CONFIG["strategy2_qi"], 1):
        k_orig = calculate_k_anonymity(df, qi_set)
        print(f"\n  QI Set {i}: {', '.join(qi_set)}")
        print(f"    Min k: {k_orig['min_k']}")
        print(f"    Avg k: {k_orig['avg_k']:.2f}")
        print(f"    Vulnerable groups: {k_orig['vulnerable_groups']}")
        
        if k_orig['min_k'] >= 10:
            print(f"    ✅ Risk: LOW")
        elif k_orig['min_k'] >= 5:
            print(f"    ⚠️  Risk: MEDIUM")
        else:
            print(f"    ❌ Risk: HIGH")
    
    print(f"\n" + "=" * 80)
    print(f"💡 RECOMMENDATIONS:")
    print("-" * 80)
    print(f"  • QI sets with min_k < 5: Apply suppression/generalization")
    print(f"  • For production: Target min_k ≥ 10")
        
except NameError:
    print("⚠️  Run Step 3 (Setup Environment) first!")

## Step 7: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports analysis results and enriched data

**What you'll see:**
- Export confirmation per strategy
- File locations and sizes
- Sample of enriched data (Strategy 3)

**Output structure:**
```
advanced_output/
├── strategy1_single_qi/output/*.json
├── strategy2_multiple_qi/output/*.json
└── strategy3_enrichment/output/enriched_data*.csv
```

**Note:** Strategy 3 includes enriched DataFrame with k-anonymity values

In [ ]:
print("=" * 80)
print("📦 EXPORT SUMMARY")
print("=" * 80)

print(f"\n📂 Export directory: {task_dir}\n")

# Strategy 1
print("Strategy 1: Single QI Analysis")
s1_dir = task_dir / 'strategy1_single_qi' / 'output'
if s1_dir.exists():
    files = list(s1_dir.glob('*.json'))
    print(f"  ✅ {len(files)} file(s) saved")
else:
    print("  ⚠️  Not available")

# Strategy 2
print("\nStrategy 2: Multiple QI Combinations")
s2_dir = task_dir / 'strategy2_multiple_qi' / 'output'
if s2_dir.exists():
    files = list(s2_dir.glob('*.json'))
    print(f"  ✅ {len(files)} file(s) saved")
else:
    print("  ⚠️  Not available")

# Strategy 3
print("\nStrategy 3: DataFrame Enrichment")
s3_dir = task_dir / 'strategy3_enrichment' / 'output'
if s3_dir.exists():
    enriched_files = list(s3_dir.glob('enriched_data*.csv'))
    if enriched_files:
        print(f"  ✅ Enriched data: {enriched_files[0].name}")
        enriched_df = pd.read_csv(enriched_files[0])
        k_cols = [col for col in enriched_df.columns if 'k_anon' in col]
        if k_cols:
            print(f"  📊 K-column: {k_cols[0]}")
            print(f"     Range: {enriched_df[k_cols[0]].min()} - {enriched_df[k_cols[0]].max()}")
    else:
        print("  ⚠️  No enriched files")
else:
    print("  ⚠️  Not available")

print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
if 'elapsed_s1' in locals():
    print(f"⏱️  Total time: {elapsed_s1 + elapsed_s2 + elapsed_s3:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 strategies implemented and compared
- ✅ Privacy metrics calculated (k-anonymity, entropy)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Single QI analysis provides detailed privacy assessment
- Multiple QI comparison identifies riskiest field combinations
- Enrichment mode enables k-value-based filtering

**Strategy recommendations:**
- **Use Strategy 1** when: Focused assessment of specific QI needed
- **Use Strategy 2** when: Comparing privacy risks across field sets
- **Use Strategy 3** when: Need k-values for downstream analysis

**Next steps:**
- Test with your own datasets
- Adjust threshold_k for privacy requirements
- Combine with anonymization operations

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)